# Imports

In [3]:
# Linear algebra
import numpy as np

# Dataframes
import pandas as pd

# Path
import os
data_path = os.path.join('..', 'data')

# Plotting
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
import matplotlib.patheffects as pe
import matplotlib.cm as cm
import matplotlib.colors as mcolors

from Bio.Data import CodonTable

# statistics
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from scipy.stats import pearsonr
from scipy.stats import ttest_ind
from scipy.stats import entropy  # KL divergence

# Fast apply
import swifter
from tqdm import tqdm

# Bio Seq
from Bio.Seq import Seq

# Load Data

## Load DataFrames

In [4]:
test_with_results = pd.read_pickle(os.path.join(data_path, '2_test_with_bart30.pkl'))

discretized_df = pd.read_pickle(os.path.join(data_path, 'genes_human_discrete_expression.pkl'))

In [5]:
# Tissue Site Detail Ids
tissueSiteDetailId = ('Adipose_Subcutaneous', 'Adipose_Visceral_Omentum', 'Adrenal_Gland', 'Artery_Aorta', 'Artery_Coronary', 'Artery_Tibial', 'Bladder', 'Brain_Amygdala', 'Brain_Anterior_cingulate_cortex_BA24', 'Brain_Caudate_basal_ganglia', 'Brain_Cerebellar_Hemisphere', 'Brain_Cerebellum', 'Brain_Cortex', 'Brain_Frontal_Cortex_BA9', 'Brain_Hippocampus', 'Brain_Hypothalamus', 'Brain_Nucleus_accumbens_basal_ganglia', 'Brain_Putamen_basal_ganglia', 'Brain_Spinal_cord_cervical_c-1', 'Brain_Substantia_nigra', 'Breast_Mammary_Tissue', 'Cells_EBV-transformed_lymphocytes', 'Cells_Cultured_fibroblasts', 'Cervix_Ectocervix', 'Cervix_Endocervix', 'Colon_Sigmoid', 'Colon_Transverse', 'Esophagus_Gastroesophageal_Junction', 'Esophagus_Mucosa', 'Esophagus_Muscularis', 'Fallopian_Tube', 'Heart_Atrial_Appendage', 'Heart_Left_Ventricle', 'Kidney_Cortex', 'Kidney_Medulla', 'Liver', 'Lung', 'Minor_Salivary_Gland', 'Muscle_Skeletal', 'Nerve_Tibial', 'Ovary', 'Pancreas', 'Pituitary', 'Prostate', 'Skin_Not_Sun_Exposed_Suprapubic', 'Skin_Sun_Exposed_Lower_leg', 'Small_Intestine_Terminal_Ileum', 'Spleen', 'Stomach', 'Testis', 'Thyroid', 'Uterus', 'Vagina', 'Whole_Blood')
temp_df = pd.DataFrame(discretized_df['median_float'].tolist(), columns=tissueSiteDetailId)

In [6]:
codon_to_fraction_path = os.path.join(data_path, "codon_to_fraction.npy")

# load .npy dictionary
codon_to_fraction = np.load(codon_to_fraction_path, allow_pickle=True).item()

# convert dictionary to DataFrame
codon_to_fraction_df = pd.DataFrame.from_dict(codon_to_fraction, orient='index', columns=['Fraction'])

## Noramlize Expression

In [7]:
# median expression floats lists
test_median_float = test_with_results['median_float'].dropna()
test_median_float_indices = test_median_float.index
test_median_float = test_median_float.tolist()
all_median_float = discretized_df['median_float'].dropna().tolist()

test_float_vectors = np.array(test_median_float)
all_float_vectors = np.array(all_median_float)

# total reads per tissue
total_reads = all_float_vectors.T.sum(axis=1)

# Normalize the float vectors (shape [num_vectors, 54]) by the total reads (shape [54])
normalized_test_float_vectors = test_float_vectors / total_reads

# Calculate Transcript Codon Biases

## Utils

In [12]:
def get_dna_codons():
  nt_bases = ['A', 'C', 'G', 'T']
  codons = [c1 + c2 + c3 for c1 in nt_bases for c2 in nt_bases for c3 in nt_bases]
  return codons

def get_amino_acids():
    codons = get_dna_codons()
    amino_acids = [Seq(codon).translate() for codon in codons]
    unique_amino_acids = list(set(amino_acids))
    return unique_amino_acids

# Create a dictionary mapping amino acids to all possible codons
def get_amino_acid_codon_dict():
  codons = get_dna_codons()
  
  amino_acid_codon_dict = {}
  for codon in codons:
    aa = str(Seq(codon).translate())
    if aa not in amino_acid_codon_dict:
      amino_acid_codon_dict[aa] = []
    amino_acid_codon_dict[aa].append(codon)
  return amino_acid_codon_dict

## Without Expression

In [ ]:
# Get the standard codon table
codon_table = CodonTable.unambiguous_dna_by_name['Standard']
# Include stop codons
codon_table.stop_codons = {'TAA', 'TAG', 'TGA'}

# Function to calculate codon fractions
def calculate_codon_fractions(sequence, codon_table):
    codon_counts = {}
    total_codons = len(sequence) // 3
    for i in range(0, len(sequence) - 2, 3):
        codon = sequence[i:i+3]
        if codon in codon_table.forward_table or codon in codon_table.stop_codons:
            codon_counts[codon] = codon_counts.get(codon, 0) + 1
    return {codon: count / total_codons for codon, count in codon_counts.items()}

# Apply the function to the 'seq' column
test_with_results['codon_fractions'] = test_with_results['seq'].apply(
    lambda seq: calculate_codon_fractions(seq, codon_table)
)

In [17]:
test_with_results['codon_fractions']

0        {'ATG': 0.022508038585209004, 'AGC': 0.0289389...
1        {'ATG': 0.028, 'TGG': 0.036, 'CTG': 0.024, 'GA...
2        {'ATG': 0.017241379310344827, 'GCC': 0.0366379...
3        {'ATG': 0.011547344110854504, 'GGG': 0.0323325...
4        {'ATG': 0.05172413793103448, 'GCT': 0.01724137...
                               ...                        
15971    {'ATG': 0.030261348005502064, 'AAC': 0.0550206...
15972    {'ATG': 0.021080368906455864, 'GGT': 0.0079051...
15973    {'ATG': 0.01948051948051948, 'GTC': 0.03246753...
15974    {'ATG': 0.03225806451612903, 'GGG': 0.03225806...
15975    {'ATG': 0.02356902356902357, 'GAC': 0.04713804...
Name: codon_fractions, Length: 15976, dtype: object

### Calculate KL-Divergence

In [ ]:
# Function to calculate KL divergence for synonymous codons of each amino acid
def calculate_kl_divergence(codon_fractions, mean_codon_fractions, codon_table):
    kl_divergences = {}
    # Reverse the mapping to group codons by their respective amino acids
    amino_acid_to_codons = {}
    for codon, amino_acid in codon_table.forward_table.items():
        if amino_acid not in amino_acid_to_codons:
            amino_acid_to_codons[amino_acid] = []
        amino_acid_to_codons[amino_acid].append(codon)

    for amino_acid, codons in amino_acid_to_codons.items():
        # Include stop codons
        if amino_acid == '*':
            codons = codon_table.stop_codons
        # Get the distributions for the synonymous codons
        codon_dist = [codon_fractions.get(codon, 0) for codon in codons]
        mean_dist = [mean_codon_fractions.get(codon, 0) for codon in codons]
        # Normalize distributions to ensure they sum to 1
        codon_dist = np.array(codon_dist) / np.sum(codon_dist) if np.sum(codon_dist) > 0 else np.zeros(len(codon_dist))
        mean_dist = np.array(mean_dist) / np.sum(mean_dist) if np.sum(mean_dist) > 0 else np.zeros(len(mean_dist))
        # Calculate KL divergence
        kl_divergences[amino_acid] = entropy(codon_dist, mean_dist)
    return kl_divergences

# Apply the function to calculate KL divergence for each row
test_with_results['kl_divergence'] = test_with_results['codon_fractions'].swifter.apply(
    lambda codon_fractions: calculate_kl_divergence(codon_fractions, codon_to_fraction, codon_table)
)

Pandas Apply: 100%|██████████| 15976/15976 [01:46<00:00, 150.39it/s]


### Calculate Amino Acid Fractions

In [22]:
amino_acids = get_amino_acids()

def amino_acid_fractions(row):
    # Create a dictionary to store the fractions of each amino acid
    aa_fractions = {}
    # Iterate through the codon fractions and calculate the fraction for each amino acid
    aa_seq = row['amino_acid_seq']
    for aa in amino_acids:
        # Count the number of codons for the amino acid in the sequence
        count = Seq(aa_seq).count(aa)
        # Calculate the fraction of the amino acid in the sequence
        fraction = count / len(aa_seq) if len(aa_seq) > 0 else 0
        aa_fractions[str(aa)] = fraction
    return aa_fractions

test_with_results['amino_acid_fractions'] = test_with_results.swifter.apply(amino_acid_fractions, axis=1)

Pandas Apply: 100%|██████████| 15976/15976 [00:00<00:00, 20647.46it/s]


In [23]:
test_with_results['amino_acid_fractions']

0        {'P': 0.07288317256162916, 'M': 0.022508038585...
1        {'P': 0.052, 'M': 0.028, 'G': 0.112, 'F': 0.03...
2        {'P': 0.09913793103448276, 'M': 0.017241379310...
3        {'P': 0.04157043879907621, 'M': 0.011547344110...
4        {'P': 0.034482758620689655, 'M': 0.05172413793...
                               ...                        
15971    {'P': 0.037138927097661624, 'M': 0.03026134800...
15972    {'P': 0.04743083003952569, 'M': 0.021080368906...
15973    {'P': 0.05194805194805195, 'M': 0.019480519480...
15974    {'P': 0.03225806451612903, 'M': 0.032258064516...
15975    {'P': 0.07744107744107744, 'M': 0.023569023569...
Name: amino_acid_fractions, Length: 15976, dtype: object

In [26]:
test_with_results.columns

Index(['cluster', 'length (aa)', 'query_gene', 'query_transcript', 'seq',
       'amino_acid_seq', 'median', 'subject_transcript', 'average_pident',
       'gene', 'transcript_x', 'median_float', 'mfc_prediction',
       'mfc_accuracy', 'mfb_prediction', 'mfb_accuracy',
       'exp_harmonized_prediction', 'exp_harmonized_accuracy', 'transcript_y',
       'bart30_prediction', 'bart30_entropy',
       'bart30_num_of_correct_predicted_codons', 'bart30_cross_entropy_loss',
       'bart30_perplexity', 'bart30_accuracy', 'bart30_ablation_prediction',
       'bart30_ablation_entropy',
       'bart30_ablation_num_of_correct_predicted_codons',
       'bart30_ablation_cross_entropy_loss', 'bart30_ablation_perplexity',
       'bart30_ablation_accuracy', 'codon_fractions', 'kl_divergence',
       'amino_acid_fractions'],
      dtype='object')

In [25]:
# Concatenate all amino acid sequences into a single sequence
combined_aa_seq = ''.join(test_with_results['amino_acid_seq'])

# Calculate the fractions for the combined sequence
def calculate_combined_amino_acid_fractions(combined_seq):
    aa_fractions = {}
    total_length = len(combined_seq)
    for aa in amino_acids:
        count = Seq(combined_seq).count(aa)
        fraction = count / total_length if total_length > 0 else 0
        aa_fractions[str(aa)] = fraction
    return aa_fractions

# Calculate the fractions
combined_amino_acid_fractions = calculate_combined_amino_acid_fractions(combined_aa_seq)

# Display the result
combined_amino_acid_fractions

{'P': 0.060517468325432304,
 'M': 0.022130431484174897,
 'G': 0.0642977743897598,
 'F': 0.0359352325203764,
 'E': 0.073150381267379,
 'V': 0.05945252265367579,
 '*': 0.0021681626165390797,
 'N': 0.035936453943983614,
 'K': 0.06016284833813783,
 'W': 0.012180579066075082,
 'Y': 0.027337903176664947,
 'C': 0.02158621941029397,
 'A': 0.06818407288044664,
 'D': 0.04831327506961775,
 'L': 0.09854988517939524,
 'Q': 0.048554302661441326,
 'H': 0.026289514580472885,
 'I': 0.044504876262324335,
 'S': 0.08162054684220604,
 'T': 0.052100638248120265,
 'R': 0.05702691108348281}

## With Expression

**Calculate Mean Codon Biases for each Tissue**

In [ ]:
# Create a dictionary mapping amino acids to list of codon frequencies
def calculate_amino_acid_codon_frac_dict(dataframe):
    amino_acid_codon_dict = get_amino_acid_codon_dict() 

    amino_acid_codon_frac_dict = {aa: [0 for _ in amino_acid_codon_dict[aa]] for aa in amino_acid_codon_dict.keys()}

    total_counts = {aa: 0 for aa in amino_acid_codon_dict.keys()}

    for row in dataframe.iterrows():
        seq = row['seq']
        codon_list = [seq[i:i+3] for i in range(0, len(seq), 3)]
        aa_seq = row['amino_acid_seq']

        for aa, codon in zip(aa_seq, codon_list):
            if codon in amino_acid_codon_dict[aa]:
                codon_index = amino_acid_codon_dict[aa].index(codon)
                amino_acid_codon_frac_dict[aa][codon_index] += 1
                total_counts[aa] += 1

    for aa, codons in amino_acid_codon_frac_dict.items():
        for i in range(len(codons)):
            if total_counts[aa] > 0:
                amino_acid_codon_frac_dict[aa][i] /= total_counts[aa]

    return amino_acid_codon_dict, amino_acid_codon_frac_dict

In [ ]:
# Calculate tissue codon biases
test_with_results_with_expression = test_with_results[test_with_results['median_float'].notna()]
aa_to_codons, tissue_biases_df = calculate_amino_acid_codon_frac_dict(test_with_results_with_expression)


In [57]:
# Get the standard codon table
codon_table = CodonTable.unambiguous_dna_by_name['Standard']

# Optimized function to calculate weighted codon fractions for each tissue using pandas vectorized operations
def calculate_weighted_codon_fractions_vectorized(sequences, expression_fractions, codon_table):
    # Create a DataFrame to store codon counts for all sequences
    codon_counts = pd.DataFrame(0, index=range(len(sequences)), columns=codon_table.forward_table.keys())
    # print shape
    print(codon_counts.shape)

    # Include stop codons in the codon table
    for stop_codon in codon_table.stop_codons:
        codon_counts[stop_codon] = 0

    # Extract codons from sequences and count them
    for idx, sequence in enumerate(sequences):
        for i in range(0, len(sequence) - 2, 3):
            codon = sequence[i:i+3]
            if codon in codon_counts.columns:
                codon_counts.at[idx, codon] += 1

    # Multiply codon counts by expression fractions
    weighted_codon_counts = codon_counts.values[:, :, None] * expression_fractions[:, None, :]

    # Reverse the mapping to group codons by their respective amino acids
    amino_acid_to_codons = {}
    for codon, amino_acid in codon_table.forward_table.items():
        if amino_acid not in amino_acid_to_codons:
            amino_acid_to_codons[amino_acid] = []
        amino_acid_to_codons[amino_acid].append(codon)
    # Include stop codons
    amino_acid_to_codons['*'] = list(codon_table.stop_codons)

    # Initialize a DataFrame to store the weighted codon fractions for each tissue
    tissue_weighted_codon_fractions = pd.DataFrame(0, index=codon_counts.columns, columns=tissueSiteDetailId)

    # Sum and normalize for each amino acid's synonymous codons
    for amino_acid, codons in amino_acid_to_codons.items():
        # Sum weighted codon counts for synonymous codons
        # Map codons to their corresponding column indices
        codon_indices = [codon_counts.columns.get_loc(codon) for codon in codons]
        codon_group_counts = weighted_codon_counts[:, codon_indices, :].sum(axis=1)
        # Normalize by total counts for the codon group
        total_codon_group_counts = codon_group_counts.sum(axis=0, keepdims=True)
        normalized_codon_group_counts = codon_group_counts / total_codon_group_counts
        # Assign normalized values back to the corresponding codons
        for i, codon in enumerate(codons):
            tissue_weighted_codon_fractions.loc[codon] = normalized_codon_group_counts[i]

    return pd.DataFrame(tissue_weighted_codon_fractions, index=codon_counts.columns, columns=tissueSiteDetailId)

# Apply the optimized function to calculate weighted codon fractions for all tissues
test_with_results_with_expression = test_with_results[test_with_results['median_float'].notna()]
weighted_codon_fractions_by_tissue = calculate_weighted_codon_fractions_vectorized(
    test_with_results_with_expression['seq'], normalized_test_float_vectors, codon_table
)

(12414, 61)


In [61]:
weighted_codon_fractions_by_tissue.sum(axis=0)

Adipose_Subcutaneous                     0.001697
Adipose_Visceral_Omentum                 0.001006
Adrenal_Gland                            0.001882
Artery_Aorta                             0.000232
Artery_Coronary                          0.000401
Artery_Tibial                            0.000302
Bladder                                  0.000226
Brain_Amygdala                           0.007475
Brain_Anterior_cingulate_cortex_BA24     0.005998
Brain_Caudate_basal_ganglia              0.007874
Brain_Cerebellar_Hemisphere              0.001491
Brain_Cerebellum                         0.002109
Brain_Cortex                             0.005622
Brain_Frontal_Cortex_BA9                 0.004167
Brain_Hippocampus                        0.005093
Brain_Hypothalamus                       0.003854
Brain_Nucleus_accumbens_basal_ganglia    0.006820
Brain_Putamen_basal_ganglia              0.007638
Brain_Spinal_cord_cervical_c-1           0.005777
Brain_Substantia_nigra                   0.005842


In [62]:
weighted_codon_fractions_by_tissue.to_pickle(os.path.join(data_path, 'weighted_codon_fractions_by_tissue.pkl'))

In [53]:
weighted_codon_fractions_by_tissue

,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,Brain_Anterior_cingulate_cortex_BA24,Brain_Caudate_basal_ganglia,...,Skin_Not_Sun_Exposed_Suprapubic,Skin_Sun_Exposed_Lower_leg,Small_Intestine_Terminal_Ileum,Spleen,Stomach,Testis,Thyroid,Uterus,Vagina,Whole_Blood
TTT,0.014723,0.015098,0.015934,0.013119,0.013541,0.012966,0.014180,0.015350,0.015287,0.015418,...,0.014073,0.013900,0.014847,0.014018,0.015556,0.015428,0.015289,0.013806,0.014633,0.012247
TTC,0.021413,0.021774,0.021692,0.021697,0.021716,0.021884,0.021730,0.020635,0.020899,0.020789,...,0.021252,0.020700,0.020904,0.020904,0.020948,0.019248,0.020597,0.021706,0.021021,0.022044
TTA,0.004818,0.004936,0.005383,0.003834,0.004067,0.003692,0.004511,0.005332,0.005316,0.005484,...,0.004467,0.004395,0.005031,0.004593,0.005231,0.006470,0.005295,0.004247,0.004795,0.003364
TTG,0.010567,0.010693,0.011502,0.009951,0.010188,0.010011,0.010295,0.010817,0.010804,0.010977,...,0.010178,0.009999,0.010618,0.010037,0.011025,0.011925,0.010867,0.010146,0.010576,0.008904
TCT,0.012934,0.012998,0.012848,0.011655,0.011897,0.011843,0.012472,0.012702,0.012550,0.012675,...,0.014127,0.014948,0.013195,0.013264,0.013099,0.014090,0.013366,0.012449,0.012735,0.013112
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GGA,0.015681,0.015701,0.015155,0.015246,0.015442,0.014974,0.015435,0.015236,0.015343,0.015434,...,0.016196,0.017525,0.015298,0.014977,0.015464,0.015532,0.015430,0.015874,0.015249,0.014484
GGG,0.017638,0.017363,0.016455,0.018200,0.017906,0.018035,0.017642,0.016530,0.016841,0.016735,...,0.017733,0.018128,0.017787,0.017837,0.016782,0.016053,0.017156,0.017975,0.017170,0.019538
TAG,0.000667,0.000665,0.000622,0.000887,0.000928,0.000948,0.000774,0.000554,0.000561,0.000544,...,0.000612,0.000604,0.000597,0.000614,0.000730,0.000572,0.000586,0.000778,0.000657,0.000684
TAA,0.000988,0.001066,0.000974,0.000962,0.001008,0.000982,0.001023,0.001075,0.000984,0.001002,...,0.000970,0.000921,0.001031,0.000948,0.001140,0.000852,0.000996,0.000995,0.001035,0.000865


**Calculate Transcript Codon Biases for each Tissue**

## Save

In [27]:
test_with_results.to_pickle(os.path.join(data_path, 'test_with_kl_divergence.pkl'))

# Graphs

In [8]:
test_with_results = pd.read_pickle(os.path.join(data_path, "test_with_kl_divergence.pkl"))

## Transcript vs Mean Codon Fraction KL-Divergence 

In [32]:
test_with_results

,cluster,length (aa),query_gene,query_transcript,seq,amino_acid_seq,median,subject_transcript,average_pident,gene,...,bart30_perplexity,bart30_accuracy,bart30_ablation_prediction,bart30_ablation_entropy,bart30_ablation_num_of_correct_predicted_codons,bart30_ablation_cross_entropy_loss,bart30_ablation_perplexity,bart30_ablation_accuracy,codon_fractions,kl_divergence
0,2191,932,ENSG00000134324,ENST00000396097,ATGAGCAGAGTGCAGACCATGAATTACGTGGGGCAGTTAGCCGGCC...,MSRVQTMNYVGQLAGQVFVTVKELYKGLNPATLSGCIDIIVIRQPN...,"(expr_pre75_90, expr_pre50_75, expr_pre75_90, ...",ENST00000261596,48.943442,ENSG00000134324,...,2.917465,0.508039,ATGAGCAGAGTGCAGACCATGAACTATGTGGGGCAGCTGGCTGGCC...,-4106.103027,469,1.070143,2.915796,0.502680,"{'ATG': 0.022508038585209004, 'AGC': 0.0289389...","{'F': 0.03355349851627153, 'L': 0.128530468048..."
1,5329,249,ENSG00000172456,ENST00000634606,ATGTGGCTGGACCATCGAGCAGTCAGTCAAGTTAACAGGATCAATG...,MWLDHRAVSQVNRINETKHSVLQYVGGVMSVEMQAPKLLWLKENLR...,"(expr_pre25_50, expr_pre25_50, expr_pre25_50, ...",NaN,0.000000,ENSG00000172456,...,3.019833,0.464000,ATGTGGCTGGACCACAGAGCTGTGTCCCAGGTGAACAGAATCAATG...,-4239.449219,116,1.117450,3.057049,0.464000,"{'ATG': 0.028, 'TGG': 0.036, 'CTG': 0.024, 'GA...","{'F': 0.0031112738749424773, 'L': 0.0820192839..."
2,7096,463,ENSG00000118495,ENST00000416623,ATGGCCACGTTCCCCTGCCAGTTATGTGGCAAGACGTTCCTCACCC...,MATFPCQLCGKTFLTLEKFTIHNYSHSRERPYKCVQPDCGKAFVSR...,"(expr_pre25_50, expr_pre50_75, expr_pre50_75, ...",ENST00000367405,18.618728,ENSG00000118495,...,2.703889,0.575431,ATGGCCACCTTCCCCTGCCAGCTCTGTGGGAAGACCTTCCTGACCC...,-4101.684570,262,1.007104,2.737661,0.564655,"{'ATG': 0.017241379310344827, 'GCC': 0.0366379...","{'F': 0.04636117096925435, 'L': 0.025243768794..."
3,7405,432,ENSG00000257950,ENST00000550383,ATGGGGCAGGCGGGCTGCAAGGGGCTCTGCCTGTCGCTGTTCGACT...,MGQAGCKGLCLSLFDYKTEKYVIAKNKKVGLLYRLLQASILAYLVV...,NaN,ENST00000413302,38.319426,ENSG00000257950,...,2.742144,0.598152,ATGGGCCAGGCTGGCTGCAAAGGCCTCTGCCTGTCTCTCTTTGACT...,-4199.959473,260,0.994005,2.702035,0.600462,"{'ATG': 0.011547344110854504, 'GGG': 0.0323325...","{'F': 0.04636117096925435, 'L': 0.170469937224..."
4,5500,57,ENSG00000107669,ENST00000690706,ATGGCTTTCTGGGCGGGGGGTTCGCCCAGCGTCGTGGACTATTTCC...,MAFWAGGSPSVVDYFPSEDFYRCGYCKNESGSRSNGMWAHSMTVQD...,NaN,NaN,0.000000,ENSG00000107669,...,2.904404,0.517241,ATGGCCTTCTGGGCTGGAGGCAGCCCCTCTGTGGTGGACTACTTCC...,-4496.814941,29,1.037383,2.821822,0.500000,"{'ATG': 0.05172413793103448, 'GCT': 0.01724137...","{'F': 0.661282805639471, 'L': 1.68159816195101..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15971,3560,726,ENSG00000081277,ENST00000367324,ATGAACCACTCGCCGCTCAAGACCGCCTTGGCGTACGAATGCTTCC...,MNHSPLKTALAYECFQDQDNSTLALPSDQKMKTGTSGRQRVQEQVM...,"(expr_pre75_90, expr_pre75_90, expr_pre25_50, ...",ENST00000331563,23.066762,ENSG00000081277,...,2.366260,0.669876,ATGAACCACTCTCCTCTGAAGACAGCCCTGGCCTATGAGTGCTTCC...,-4110.647949,457,0.968862,2.634945,0.628611,"{'ATG': 0.030261348005502064, 'AAC': 0.0550206...","{'F': 0.30669314272446774, 'L': 0.276568366436..."
15972,3464,758,ENSG00000105298,ENST00000429344,ATGGGTCGGGACACACGCTCGCGCTCGCGGTCCGCGGGTCGCCGGG...,MGRDTRSRSRSAGRRGRRRQSQSGSRSRSRSHGRRNRRRREDEGRR...,"(expr_pre75_90, expr_pre75_90, expr_pre75_90, ...",NaN,0.000000,ENSG00000105298,...,2.345779,0.683794,ATGGGCCGGGACACCCGCAGCCGGAGCCGCAGCGCGGGCCGCCGGG...,-4132.578613,472,0.983526,2.673868,0.621871,"{'ATG': 0.021080368906455864, 'GGT': 0.0079051...","{'F': 0.3277234844046498, 'L': 0.5402330541438..."
15973,2613,153,ENSG00000066468,ENST00000683035,ATGGTCAGCTGGGGTCGTTTCATCTGCCTGGTCGTGGTCACCATGG...,MVSWGRFICLVVVTMATLSLARPSFSLVEDTTLEPEEPPTKYQISQ...,NaN,ENST00000393637,21.568418,ENSG00000066468,...,3.004799,0.487013,ATGGTGTCCTGGGGCCGCTTCATCTGCCTGGTGGTGGTGACCATGG...,-4170.503418,77,1.099506,3.002682,0.500000,"{'ATG': 0.01948051948051948, 'GTC': 0.03246753...","{'F': 0.11514206123910978, 'L': 0.209593394298..."
15974,530,61,ENSG00000197892,ENST00000522355,ATGGGGGACTCCAAAGT

In [11]:
# ignore nan values in np.mean([1, np.nan, 3])
# Example usage
np.nanmean([1, np.nan, 3])

np.float64(2.0)